# 06. Proces B1 — stan upraw względem normy 2001–2017

Dla każdego województwa i każdego terminu MODIS (co 16 dni) porównujemy średni NDVI **pól uprawnych** z normą z lat 2001–2017. Wynik: status *powyżej / w normie / poniżej normy*, a dodatkowo ocena całego sezonu IV–IX.

Przepływ: granice województw (GUGiK ULDK, notatnik 05) → NDVI MODIS + maska pól WorldCover (notatnik 02) → średnia per region i termin → uzupełnienie luk + wygładzenie (Savitzky–Golay) → norma 2001–2017 → z-score → status.

Wymagane: `pip install rasterio geopandas shapely pyarrow scipy matplotlib` oraz `lib/ndvi_tools.py`. Bez konta.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from shapely import wkt

import sys

# Katalog główny repo: notatnik może być uruchomiony z notebooks/ albo z głównego folderu
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from lib import ndvi_tools as nt

ULDK = "https://uldk.gugik.gov.pl/"
DATA = ROOT / "data"          # wspólny folder danych w katalogu głównym (jest w .gitignore)
DATA.mkdir(exist_ok=True)

START, END = 2001, 2024
DECIMATION = 4        # 4 = szybki podgląd (co 4. piksel); do wyników końcowych ustawić 1
Z_THRESHOLD = 1.0     # |z| >= 1 odchylenia standardowego normy => powyżej / poniżej normy
TEST_YEARS = (2018, 2019)   # susze testowe, poza normą

## Granice województw

Jak w notatniku 05 (ULDK, `srid=4326`). Wynik zapisujemy lokalnie, żeby nie odpytywać GUGiK przy każdym uruchomieniu. `region_id` = kod TERYT.

In [ ]:
def load_voivodeships(cache=DATA / "granice_wojewodztwa.gpkg") -> gpd.GeoDataFrame:
    if cache.exists():
        return gpd.read_file(cache)
    rows = []
    for code in [f"{n:02d}" for n in range(2, 33, 2)]:
        params = {"request": "GetVoivodeshipById", "id": code,
                  "result": "teryt,voivodeship,geom_wkt", "srid": 4326}
        resp = requests.get(ULDK, params=params, timeout=60)
        resp.raise_for_status()
        lines = resp.text.strip().split("\n")
        if lines[0].strip() != "0":
            raise RuntimeError(f"ULDK {code}: {resp.text[:200]}")
        teryt, name, geom = lines[1].split("|")
        rows.append({"region_id": teryt, "name": name, "geometry": wkt.loads(geom.split(";", 1)[1])})
    gdf = gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:4326")
    gdf.to_file(cache, driver="GPKG")
    return gdf


regions = load_voivodeships()
ax = regions.plot(edgecolor="black", facecolor="none", figsize=(7, 7))
for _, r in regions.iterrows():
    p = r.geometry.representative_point()
    ax.annotate(r["name"], (p.x, p.y), ha="center", fontsize=7)
plt.show()

## NDVI pól per województwo

`build_regional_ndvi` robi całość: katalog kafelków MOD13Q1 → odczyt NDVI + maska `pixel_reliability` (0, 1) → maska pól (WorldCover klasa 40, próg 50% piksela) → średnia per województwo. Wynik zapisuje się do parquet **per rok** i kolejne uruchomienia pomijają gotowe lata (pełny zakres to ok. 100 GB do odczytu, więc pierwsze uruchomienie jest długie).

Katalog wyników zawiera `DECIMATION` w nazwie, żeby nie mieszać podglądu z danymi pełnymi. Przy zmianie `regions` zmienić `cache_tag`.

In [ ]:
regional = nt.build_regional_ndvi(
    regions, START, END,
    decimation=DECIMATION,
    out_dir=DATA / f"ndvi_regional_d{DECIMATION}",
    cache_dir=DATA / "grids",
    cache_tag="woj",
    max_workers=4,
)
print(regional.shape, "| regiony:", regional["region_id"].nunique())
regional.head()

Kontrola jakości: każdy rok powinien mieć 23 terminy; `brak_ndvi` to udział terminów, w których pokrycie ważnymi pikselami było poniżej progu (chmury, śnieg). Zimą wartości będą wysokie, to normalne — luki uzupełnia kolejny krok.

In [ ]:
qc = regional.groupby("year").agg(
    terminy=("date", "nunique"),
    brak_ndvi=("ndvi_mean", lambda s: s.isna().mean()),
    srednie_pokrycie=("coverage", "mean"),
).round(3)
qc

## B1: porównanie z normą

`run_b1` wykonuje: pełna siatka region × rok × termin → uzupełnienie luk (do 3 kolejnych terminów) → SG (okno 5, wielomian 2) → norma 2001–2017 per `(województwo, termin)` → `diff`, `pct_of_norm`, `z`, `status`. Dodatkowo ocena sezonu IV–IX (średni NDVI) względem normy sezonowej.

Normy nie liczymy z lat 2018–2024 (testowane susze) — funkcja `compute_norm` rzuca błąd, gdyby ktoś próbował.

In [ ]:
periods, season = nt.run_b1(regional, z_threshold=Z_THRESHOLD)

names = regions.set_index("region_id")["name"]
for df in (periods, season):
    df["region"] = df["region_id"].map(names)

season.head()

## Susze testowe 2018 i 2019 (sezon IV–IX)

In [ ]:
test = season[season["year"].isin(TEST_YEARS)]
pd.concat({
    "% normy": test.pivot(index="region", columns="year", values="pct_of_norm").round(1),
    "z": test.pivot(index="region", columns="year", values="z").round(2),
    "status": test.pivot(index="region", columns="year", values="status"),
}, axis=1)

In [ ]:
z = season.pivot(index="region", columns="year", values="z")
fig, ax = plt.subplots(figsize=(13, 5))
im = ax.imshow(z.values, cmap="RdYlGn", vmin=-3, vmax=3, aspect="auto")
ax.set_xticks(range(len(z.columns)))
ax.set_xticklabels(z.columns, rotation=90)
ax.set_yticks(range(len(z.index)))
ax.set_yticklabels(z.index)
for lbl in ax.get_xticklabels():
    if int(lbl.get_text()) in TEST_YEARS:
        lbl.set_color("red")
if 2017 in z.columns:
    ax.axvline(list(z.columns).index(2017) + 0.5, color="black", ls="--", lw=1)   # koniec lat normy
fig.colorbar(im, ax=ax, label="z (sezon IV–IX vs norma 2001–2017)")
ax.set_title("B1: anomalia NDVI sezonu (zielony = powyżej normy, czerwony = poniżej)")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(TEST_YEARS), figsize=(6 * len(TEST_YEARS), 6))
for ax, y in zip(np.atleast_1d(axes), TEST_YEARS):
    d = regions.merge(season.loc[season["year"] == y, ["region_id", "z"]], on="region_id")
    d.plot(column="z", cmap="RdYlGn", vmin=-3, vmax=3, edgecolor="black", linewidth=0.5,
           legend=True, ax=ax, missing_kwds={"color": "lightgrey"})
    ax.set_title(f"B1: sezon IV–IX {y} (z względem normy)")
    ax.set_axis_off()
plt.show()

## Krzywa roku vs norma

Pasmo = norma 2001–2017 (średnia ± 1 odch. std.) dla wybranego województwa; linie = lata testowe.

In [ ]:
REGION = "30"   # TERYT: 30 = wielkopolskie

cur = periods[periods["region_id"] == REGION]
norm = cur[cur["year"] == cur["year"].min()].sort_values("period")
x = norm["period"] * 16 + 1

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.fill_between(x, norm["norm_mean"] - norm["norm_std"], norm["norm_mean"] + norm["norm_std"],
                color="lightgrey", label="norma 2001–2017 ± 1 std")
ax.plot(x, norm["norm_mean"], color="black", lw=1, label="norma (średnia)")
for y in TEST_YEARS:
    c = cur[cur["year"] == y].sort_values("period")
    ax.plot(c["period"] * 16 + 1, c["ndvi_smooth"], marker="o", ms=3, label=str(y))
ax.set_xlabel("dzień roku (początek kompozytu 16-dniowego)")
ax.set_ylabel("NDVI pól (wygładzone)")
ax.set_title(f"{names[REGION]}: krzywa NDVI vs norma")
ax.legend()
plt.show()

## Kontrola wyników

Udział województw ze statusem „poniżej normy” w sezonie. W latach normy przy progu |z| ≥ 1 naturalnie będzie to rzędu kilkunastu procent; lata testowe (susze) powinny wyraźnie wyróżniać się w górę.

Porównanie z European Drought Observatory (stan roślinności, anomalie 2018–2019, wg README sekcja 7) wymaga plików z EDO, które są jeszcze „do ustalenia” — tu tylko kontrola wewnętrzna.

In [ ]:
share = (season.assign(below=season["status"] == "poniżej normy")
               .groupby("year")["below"].mean())

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(share.index, share.values, color=["tab:red" if y in TEST_YEARS else "tab:gray" for y in share.index])
ax.set_ylabel("udział województw poniżej normy")
plt.show()

print(f"średnio w latach normy (<= 2017): {share[share.index <= 2017].mean():.0%}")
for y in TEST_YEARS:
    if y in share.index:
        print(f"{y}: {share[y]:.0%}")

## Eksport

Dwie tabele do dalszych kroków (ładowanie do ClickHouse / Grafany): `b1_okresy` (województwo × rok × termin) i `b1_sezon` (województwo × rok).

In [ ]:
out = DATA / "b1"
out.mkdir(exist_ok=True)
periods.to_parquet(out / "b1_okresy.parquet", index=False)
season.to_parquet(out / "b1_sezon.parquet", index=False)
print(len(periods), "wierszy okresów,", len(season), "wierszy sezonów ->", out)